# Eksperimen Ekstraksi Fitur NLP (Ablation Study) — Regresi Log Return

**Update:** target eksperimen ini sudah diubah dari klasifikasi biner
NAIK/TURUN (versi awal) menjadi **regresi kontinu log return**:

```
Target_t = log( KursTengah_(t+1) / KursTengah_t )
```

Split, feature engineering historis, dan cara fit TF-IDF tetap sama seperti
versi awal (kronologis 70/15/15, TF-IDF di-fit hanya di train). Yang
berubah: jenis target, metrik evaluasi (RMSE & MAE ditambah), dan model
yang dipakai (ElasticNet, LightGBM, XGBoost — bukan Logistic Regression,
karena targetnya sudah kontinu).

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.append("../src")

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNetCV
from xgboost import XGBRegressor

from nlp_feature_extraction import (
    LEXICON_FEATURES,
    TS_FEATURES,
    TfidfSvdExtractor,
    add_lexicon_features,
    chronological_split,
    evaluate_model,
    evaluate_naive,
    load_baseline_df_model,
    load_daily_news,
    train_eval_regressor,
)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

## 1. Rekonstruksi split + gabungkan fitur NLP harian

Sama seperti versi awal: baris & split direproduksi dari kerangka baseline,
fitur NLP harian (dari `data/processed/merge_data.csv`) di-*left join*
berdasarkan `Tanggal_dt`. Bedanya di sini: `Target` sekarang log return
kontinu, bukan label 0/1.

In [2]:
df_model = load_baseline_df_model(RAW_DIR)
news_daily = load_daily_news(PROCESSED_DIR)
news_daily = add_lexicon_features(news_daily)

full = df_model.merge(news_daily, on="Tanggal_dt", how="left")
assert len(full) == len(df_model), "jumlah baris berubah setelah merge -- cek key tanggal!"

full[LEXICON_FEATURES] = full[LEXICON_FEATURES].fillna(0.0)
full["all_text_lemmatized"] = full["all_text_lemmatized"].fillna("")

print(f"Total baris: {len(full)}")
print(f"Target (log return) -- mean: {full['Target'].mean():.6f}, std: {full['Target'].std():.6f}")
print(f"Proporsi hari return positif (setara 'NAIK' di versi klasifikasi): {(full['Target'] > 0).mean():.3f}")
full[["Tanggal_dt", "Kurs Tengah", "Target", "naive_last_return", "news_count", "has_news"]].head()

Total baris: 1196
Target (log return) -- mean: 0.000187, std: 0.003340
Proporsi hari return positif (setara 'NAIK' di versi klasifikasi): 0.556


,Tanggal_dt,Kurs Tengah,Target,naive_last_return,news_count,has_news
0,2021-09-08,14195.005,0.004989,-0.003095,1.0,1
1,2021-09-09,14266.000,0.000420,0.004989,1.0,1
2,2021-09-10,14272.000,-0.003298,0.000420,1.0,1
3,2021-09-13,14225.005,0.002457,-0.003298,4.0,1
4,2021-09-14,14260.000,-0.000210,0.002457,2.0,1


In [3]:
train_df, val_df, test_df = chronological_split(full)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Periode train : {train_df['Tanggal_dt'].min().date()} s.d. {train_df['Tanggal_dt'].max().date()}")
print(f"Periode test  : {test_df['Tanggal_dt'].min().date()} s.d. {test_df['Tanggal_dt'].max().date()}")

Train: 837 | Val: 179 | Test: 180
Periode train : 2021-09-08 s.d. 2025-02-19
Periode test  : 2025-11-24 s.d. 2026-08-31


## 2. Naive Baselines (regresi)

Dua pembanding *naive*, versi regresi dari `Naive Majority Class` /
`Naive Persistence Arah` di eksperimen klasifikasi awal:

- **Naive Zero Return**: prediksi return besok = 0 (asumsi random-walk /
  efficient market -- tidak ada perubahan yang diharapkan).
- **Naive Last Return**: prediksi return besok = return yang baru saja
  terealisasi hari ini (`naive_last_return`, dihitung sebelum split supaya
  tidak salah ambil baris di seberang batas train/val/test).

In [4]:
results = []

rows_zero = []
for split_name, split_df in [("Val", val_df), ("Test", test_df)]:
    zero_pred = np.zeros(len(split_df))
    rows_zero.append(evaluate_model(split_df["Target"], zero_pred, f"Naive Zero Return - {split_name}"))
results += rows_zero

rows_last = evaluate_naive(train_df, val_df, test_df, "naive_last_return", "Naive Last Return")
results += rows_last

pd.DataFrame(rows_zero + rows_last)

,model,RMSE,MAE,directional_accuracy,balanced_directional_accuracy
0,Naive Zero Return - Val,0.003374,0.002464,0.474860,0.500000
1,Naive Zero Return - Test,0.002908,0.002205,0.422222,0.500000
2,Naive Last Return - Val,0.004600,0.003459,0.513966,0.512453
3,Naive Last Return - Test,0.003977,0.003084,0.550000,0.539727


## 3. Fitur TF-IDF/SVD per skenario

Sama seperti versi awal: `TfidfSvdExtractor` di-`fit` **hanya di
`all_text_lemmatized` milik `train_df`**, lalu `.transform()` saja untuk
val/test.

In [5]:
extractor_b = TfidfSvdExtractor(n_components=3, min_df=5)
extractor_b.fit(train_df["all_text_lemmatized"])
print(f"Skenario B -- explained variance (3 komponen): {extractor_b.explained_variance_ratio_.sum():.3f}")

extractor_c = TfidfSvdExtractor(n_components=20, min_df=5)
extractor_c.fit(train_df["all_text_lemmatized"])
print(f"Skenario C -- explained variance (20 komponen): {extractor_c.explained_variance_ratio_.sum():.3f}")

def attach_svd(extractor, df):
    svd = extractor.transform(df["all_text_lemmatized"])
    return pd.concat([df.reset_index(drop=True), svd.reset_index(drop=True)], axis=1), list(svd.columns)

train_b, svd_b_cols = attach_svd(extractor_b, train_df)
val_b, _ = attach_svd(extractor_b, val_df)
test_b, _ = attach_svd(extractor_b, test_df)

train_c, svd_c_cols = attach_svd(extractor_c, train_df)
val_c, _ = attach_svd(extractor_c, val_df)
test_c, _ = attach_svd(extractor_c, test_df)

feature_sets = {
    "TS-only": (TS_FEATURES, train_df, val_df, test_df),
    "Skenario A (Lexicon)": (TS_FEATURES + LEXICON_FEATURES, train_df, val_df, test_df),
    "Skenario B (+TF-IDF 3)": (TS_FEATURES + LEXICON_FEATURES + svd_b_cols, train_b, val_b, test_b),
    "Skenario C (+TF-IDF 20)": (TS_FEATURES + LEXICON_FEATURES + svd_c_cols, train_c, val_c, test_c),
}
print("Feature set siap:", {k: len(v[0]) for k, v in feature_sets.items()})

Skenario B -- explained variance (3 komponen): 0.079


Skenario C -- explained variance (20 komponen): 0.192


Feature set siap: {'TS-only': 7, 'Skenario A (Lexicon)': 12, 'Skenario B (+TF-IDF 3)': 15, 'Skenario C (+TF-IDF 20)': 32}


## 4. Latih 3 model baru × 4 skenario fitur

- **ElasticNetCV** (`l1_ratio` dicoba [0.1, 0.5, 0.9], `alpha` dipilih
  otomatis lewat 5-fold CV di TRAIN saja) -- fitur di-scale dulu
  (`StandardScaler`, fit di train).
- **LightGBM** (`LGBMRegressor`, default wajar: 300 pohon, depth 4,
  learning rate 0.03) -- tidak perlu scaling.
- **XGBoost** (`XGBRegressor`, setting sama dengan LightGBM untuk
  perbandingan yang adil) -- tidak perlu scaling.

In [6]:
def make_models():
    return {
        "ElasticNet": (ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, random_state=42, max_iter=5000), True),
        "LightGBM": (LGBMRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, random_state=42, verbosity=-1), False),
        "XGBoost": (XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, random_state=42), False),
    }

fitted_models = {}
for scenario_name, (cols, tr, va, te) in feature_sets.items():
    for model_name, (model, scale) in make_models().items():
        full_name = f"{model_name} [{scenario_name}]"
        rows, fitted = train_eval_regressor(model, cols, tr, va, te, full_name, scale=scale)
        results += rows
        fitted_models[full_name] = fitted

print(f"Total kombinasi model x skenario yang dilatih: {len(feature_sets) * len(make_models())}")

Total kombinasi model x skenario yang dilatih: 12


## 5. Cek ElasticNet: berapa banyak fitur yang benar-benar dipakai?

Diagnostik cepat -- ElasticNetCV bisa saja meregularisasi semua koefisien
jadi 0 kalau memang tidak ada sinyal linear yang lolos cross-validation.

In [7]:
for scenario_name in feature_sets:
    full_name = f"ElasticNet [{scenario_name}]"
    model = fitted_models[full_name]
    n_nonzero = int(np.sum(model.coef_ != 0))
    n_total = len(model.coef_)
    print(f"{full_name}: alpha={model.alpha_:.5f}, l1_ratio={model.l1_ratio_}, koefisien nonzero={n_nonzero}/{n_total}")

ElasticNet [TS-only]: alpha=0.00173, l1_ratio=0.1, koefisien nonzero=0/7
ElasticNet [Skenario A (Lexicon)]: alpha=0.00191, l1_ratio=0.1, koefisien nonzero=0/12
ElasticNet [Skenario B (+TF-IDF 3)]: alpha=0.00191, l1_ratio=0.1, koefisien nonzero=0/15
ElasticNet [Skenario C (+TF-IDF 20)]: alpha=0.00026, l1_ratio=0.9, koefisien nonzero=3/32


## 6. Tabel Perbandingan Lengkap (regresi)

In [8]:
summary = pd.DataFrame(results)
summary["dataset"] = summary["model"].apply(lambda x: "Val" if x.endswith("Val") else "Test")
summary["model"] = summary["model"].str.replace(" - Val", "", regex=False).str.replace(" - Test", "", regex=False)

pivot = summary.pivot(index="model", columns="dataset", values=["RMSE", "MAE", "directional_accuracy", "balanced_directional_accuracy"])
pivot.columns = [f"{metric}_{ds.lower()}" for metric, ds in pivot.columns]
pivot = pivot[[
    "RMSE_val", "RMSE_test", "MAE_val", "MAE_test",
    "directional_accuracy_val", "directional_accuracy_test",
    "balanced_directional_accuracy_val", "balanced_directional_accuracy_test",
]].sort_values("RMSE_test")

best_naive_rmse_test = min(
    r["RMSE"] for r in (rows_zero + rows_last) if r["model"].endswith("Test")
)
pivot["selisih_rmse_vs_naive_terbaik"] = pivot["RMSE_test"] - best_naive_rmse_test

output_path = Path("nlp_experiment_results_regression.csv")
pivot.to_csv(output_path)
print(f"Tabel hasil regresi disimpan ke: {output_path.resolve()}")
pivot

Tabel hasil regresi disimpan ke: /Users/farsyaant/nlpproject-grestercinta/notebook/nlp_experiment_results_regression.csv


,RMSE_val,RMSE_test,MAE_val,MAE_test,directional_accuracy_val,directional_accuracy_test,balanced_directional_accuracy_val,balanced_directional_accuracy_test,selisih_rmse_vs_naive_terbaik
model,,,,,,,,,
ElasticNet [Skenario A (Lexicon)],0.003372,0.002893,0.002459,0.002180,0.525140,0.577778,0.500000,0.500000,-0.000014
ElasticNet [Skenario B (+TF-IDF 3)],0.003372,0.002893,0.002459,0.002180,0.525140,0.577778,0.500000,0.500000,-0.000014
ElasticNet [TS-only],0.003372,0.002893,0.002459,0.002180,0.525140,0.577778,0.500000,0.500000,-0.000014
ElasticNet [Skenario C (+TF-IDF 20)],0.003371,0.002898,0.002453,0.002191,0.536313,0.572222,0.512328,0.495192,-0.000009
Naive Zero Return,0.003374,0.002908,0.002464,0.002205,0.474860,0.422222,0.500000,0.500000,0.000000
LightGBM [TS-only],0.003493,0.003008,0.002672,0.002288,0.513966,0.494444,0.515269,0.496964,0.000100
XGBoost [Skenario A (Lexicon)],0.003584,0.003094,0.002637,0.002312,0.463687,0.461111,0.467397,0.455719,0.000186
LightGBM [Skenario C (+TF-IDF 20)],0.003453,0.003147,0.002529,0.002442,0.558659,0.455556,0.555569,0.433198,0.000239
XGBoost [Skenario C (+TF-IDF 20)],0.003409,0.003181,0.002502,0.002455,0.564246,0.444444,0.568210,0.448381,0.000274


## 7. Insight (regresi log return)

In [9]:
print(pivot.to_string())

                                      RMSE_val  RMSE_test   MAE_val  MAE_test  directional_accuracy_val  directional_accuracy_test  balanced_directional_accuracy_val  balanced_directional_accuracy_test  selisih_rmse_vs_naive_terbaik
model                                                                                                                                                                                                                                   
ElasticNet [Skenario A (Lexicon)]     0.003372   0.002893  0.002459  0.002180                  0.525140                   0.577778                           0.500000                            0.500000                      -0.000014
ElasticNet [Skenario B (+TF-IDF 3)]   0.003372   0.002893  0.002459  0.002180                  0.525140                   0.577778                           0.500000                            0.500000                      -0.000014
ElasticNet [TS-only]                  0.003372   0.002893  0.002459 

**Tidak ada model/skenario yang mengungguli baseline naive secara berarti.**

- **ElasticNetCV meregularisasi HABIS semua fitur jadi 0** di 3 dari 4
  skenario (TS-only: 0/7, Skenario A: 0/12, Skenario B: 0/15 koefisien
  nonzero) -- artinya cross-validation sendiri menyimpulkan tidak ada
  sinyal linear yang cukup kuat untuk dipakai, dan model berakhir cuma
  memprediksi rata-rata return historis untuk SEMUA hari, terlepas dari
  fitur historis maupun fitur beritanya. RMSE test-nya (0.0028934) memang
  jadi yang terendah di tabel, tapi itu murni properti matematis dari
  memprediksi rata-rata sampel (selalu meminimalkan MSE in-sample lebih
  baik dari menebak 0 persis) -- **bukan bukti model belajar sesuatu dari
  data**. Selisihnya dengan Naive Zero Return cuma 0.0000144 (0.5% relatif),
  jauh di dalam rentang yang bisa dianggap noise.
- **Skenario C (32 fitur, termasuk TF-IDF 20 komponen) satu-satunya yang
  menyisakan koefisien nonzero (3/32)** -- tapi RMSE test-nya malah
  sedikit LEBIH BURUK (0.0028984) daripada versi 0-koefisien di atasnya.
  Menambah dimensi fitur di sini tidak menang apa-apa.
- **LightGBM dan XGBoost (model non-linear) justru semuanya KALAH dari
  kedua naive baseline** dalam RMSE/MAE test, di semua 4 skenario fitur.
  Ini konsisten dengan temuan overfitting yang sama seperti eksperimen
  klasifikasi sebelumnya: dengan hanya 837 baris data train dan sinyal
  yang sangat lemah, model berkapasitas tinggi (tree ensemble) lebih
  mudah menghafal noise di training set daripada penurunan performa
  konsisten sudah dua kali muncul di dua formulasi target yang berbeda
  (klasifikasi maupun regresi).
- **Directional accuracy tidak menyelamatkan cerita ini.** Beberapa
  kombinasi (mis. LightGBM/XGBoost Skenario C) sempat terlihat oke di Val
  (dir. acc ~0.56-0.58) tapi anjlok di Test (~0.44-0.46) -- pola klasik
  overfit-ke-periode-validasi, bukan sinyal yang robust.
- **Kesimpulan jujur:** baik reformulasi ke regresi log return maupun
  penambahan 3 model baru (ElasticNet, LightGBM, XGBoost) tidak mengubah
  kesimpulan dari eksperimen klasifikasi sebelumnya -- fitur NLP (lexicon
  maupun TF-IDF, di granularitas blob-harian) tidak menambah nilai
  prediktif yang bisa dibuktikan pada dataset dan horizon 1-hari ini.
  Baseline paling "aman" tetap yang paling naif secara matematis (predict
  the mean / predict zero), bukan karena naive itu pintar, tapi karena
  semua model yang lebih kompleks gagal mengalahkannya secara meyakinkan.

## 8. [ARSIP] Hasil eksperimen versi awal (klasifikasi NAIK/TURUN)

Disimpan sebagai referensi historis -- **bukan dibandingkan langsung**
dengan tabel regresi di atas karena jenis target berbeda total (label 0/1
vs return kontinu). Ini angka yang sama seperti sebelum update ini.

In [10]:
baseline_results_classification = [
    {"model": "Naive (Majority Class)", "dataset": "Val", "directional_accuracy": 0.525140, "f1_score": 0.688645},
    {"model": "Naive (Majority Class)", "dataset": "Test", "directional_accuracy": 0.577778, "f1_score": 0.732394},
    {"model": "Naive (Persistence Arah)", "dataset": "Val", "directional_accuracy": 0.513966, "f1_score": 0.539683},
    {"model": "Naive (Persistence Arah)", "dataset": "Test", "directional_accuracy": 0.550000, "f1_score": 0.608696},
    {"model": "Baseline: Logistic Regression (TS-only)", "dataset": "Val", "directional_accuracy": 0.547486, "f1_score": 0.649351},
    {"model": "Baseline: Logistic Regression (TS-only)", "dataset": "Test", "directional_accuracy": 0.544444, "f1_score": 0.549451},
    {"model": "Skenario A: TS + Lexicon", "dataset": "Val", "directional_accuracy": 0.497207, "f1_score": 0.608696},
    {"model": "Skenario A: TS + Lexicon", "dataset": "Test", "directional_accuracy": 0.527778, "f1_score": 0.508671},
    {"model": "Skenario B: TS + Lexicon + TF-IDF(3)", "dataset": "Val", "directional_accuracy": 0.480447, "f1_score": 0.563380},
    {"model": "Skenario B: TS + Lexicon + TF-IDF(3)", "dataset": "Test", "directional_accuracy": 0.511111, "f1_score": 0.500000},
    {"model": "Skenario C: TS + Lexicon + TF-IDF(20)", "dataset": "Val", "directional_accuracy": 0.525140, "f1_score": 0.535519},
    {"model": "Skenario C: TS + Lexicon + TF-IDF(20)", "dataset": "Test", "directional_accuracy": 0.488889, "f1_score": 0.386667},
]
pd.DataFrame(baseline_results_classification)

,model,dataset,directional_accuracy,f1_score
0,Naive (Majority Class),Val,0.525140,0.688645
1,Naive (Majority Class),Test,0.577778,0.732394
2,Naive (Persistence Arah),Val,0.513966,0.539683
3,Naive (Persistence Arah),Test,0.550000,0.608696
4,Baseline: Logistic Regression (TS-only),Val,0.547486,0.649351
5,Baseline: Logistic Regression (TS-only),Test,0.544444,0.549451
6,Skenario A: TS + Lexicon,Val,0.497207,0.608696
7,Skenario A: TS + Lexicon,Test,0.527778,0.508671
8,Skenario B: TS + Lexicon + TF-IDF(3),Val,0.480447,0.563380
9,Skenario B: TS + Lexicon + TF-IDF(3),Test,0.511111,0.500000
